# Engenharia de Freatures: The Movies Dataset

In [29]:
# Configuração do Jupyter (Autoreload)
%load_ext autoreload
%autoreload 2

# Configuração de Caminho (Path Setup)
import sys
import os

# Adiciona a pasta raiz do projeto (..) ao sistema para liberar os imports locais
sys.path.append(os.path.abspath(os.path.join('..')))


# Importação de Bibliotecas e Módulos
import ast
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Nossos módulos customizados da pasta src/
from src import load_data_csv

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


---
## Carregando Dados

In [30]:
caminho = '../data/processed/movies_cleaned.csv'
df_movies = load_data_csv(caminho)

Dados carregados com sucesso! Formato: (41184, 18)


---
## Editando dados (pontuais) desatualizados

In [31]:
# Dicionário Expandido: Mapeamento das Maiores Franquias do Cinema 
correcoes_franquias = {
    # 🦸‍♂️ Heróis (Marvel & DC)
    'Avengers': 'The Avengers Collection',
    'Vingadores': 'The Avengers Collection',
    'Spider-Man': 'Spider-Man Collection',
    'Homem-Aranha': 'Spider-Man Collection',
    'Iron Man': 'Iron Man Collection',
    'Homem de Ferro': 'Iron Man Collection',
    'Captain America': 'Captain America Collection',
    'Thor': 'Thor Collection',
    'X-Men': 'X-Men Collection',
    'Batman': 'Batman Collection',
    'Dark Knight': 'Batman Collection',
    'Cavaleiro das Trevas': 'Batman Collection',
    'Superman': 'Superman Collection',
    
    # 🧙‍♂️ Sci-Fi & Fantasia
    'Star Wars': 'Star Wars Collection',
    'Harry Potter': 'Harry Potter Collection',
    'Lord of the Rings': 'The Lord of the Rings Collection',
    'Senhor dos Anéis': 'The Lord of the Rings Collection',
    'Hobbit': 'The Hobbit Collection',
    'Matrix': 'The Matrix Collection',
    'Hunger Games': 'The Hunger Games Collection',
    'Jogos Vorazes': 'The Hunger Games Collection',
    'Twilight': 'The Twilight Saga Collection',
    'Crepúsculo': 'The Twilight Saga Collection',
    'Jumanji': 'Jumanji Collection',

    # 💥 Ação, Aventura & Sci-Fi
    'James Bond': 'James Bond Collection',
    '007': 'James Bond Collection',
    'Fast and Furious': 'The Fast and the Furious Collection',
    'Velozes e Furiosos': 'The Fast and the Furious Collection',
    'Mission: Impossible': 'Mission: Impossible Collection',
    'Missão: Impossível': 'Mission: Impossible Collection',
    'Indiana Jones': 'Indiana Jones Collection',
    'Pirates of the Caribbean': 'Pirates of the Caribbean Collection',
    'Piratas do Caribe': 'Pirates of the Caribbean Collection',
    'Jurassic': 'Jurassic Park Collection', 
    'Terminator': 'The Terminator Collection',
    'Exterminador do Futuro': 'The Terminator Collection',
    'Transformers': 'Transformers Collection',
    'Mad Max': 'Mad Max Collection',

    # 🎬 Animações Gigantes
    'Toy Story': 'Toy Story Collection',
    'Shrek': 'Shrek Collection',
    'Despicable Me': 'Despicable Me Collection',
    'Meu Malvado Favorito': 'Despicable Me Collection',
    'Minions': 'Despicable Me Collection',
    'Ice Age': 'Ice Age Collection',
    'A Era do Gelo': 'Ice Age Collection'
}

In [32]:
# O Loop Inteligente: Varrer o dicionário e aplicar a regra
for palavra_chave, nome_franquia in correcoes_franquias.items():
    filtro = df_movies['title'].str.contains(palavra_chave, case=False, na=False)
    df_movies.loc[filtro & df_movies['belongs_to_collection'].isna(), 'belongs_to_collection'] = nome_franquia
    df_movies.loc[filtro, 'is_franchise'] = True

print(f"Correções manuais aplicadas para {len(correcoes_franquias)} chaves de franquias!")

Correções manuais aplicadas para 45 chaves de franquias!


---
## Iniciando alterações nas Colunas `str` 

In [33]:
# Função para extrair os nomes dos diretores, atores e gêneros a partir do formato JSON/Dicionário

def extrair_nomes(texto):
    # Se o valor for nulo, retorna uma lista vazia
    if pd.isna(texto):
        return []
    
    try:
        # ast.literal_eval converte a string "[]" em uma lista real do Python
        lista_dicionarios = ast.literal_eval(texto)
        
        # Extrai apenas o 'name' de cada dicionário dentro da lista
        return [item['name'] for item in lista_dicionarios]
    
    except (ValueError, SyntaxError):
        # Se der erro na conversão de algum registro corrompido, ignora e retorna vazio
        return []

In [34]:
# Aplicando a função nas colunas complexas
colunas_para_extrair = ['genres', 'production_countries', 'spoken_languages']

for coluna in colunas_para_extrair:
    # O .apply joga a nossa função linha por linha na coluna
    df_movies[coluna] = df_movies[coluna].apply(extrair_nomes)

# Visualizando o resultado limpo
display(df_movies[['title', 'genres', 'production_countries', 'spoken_languages']].head())

,title,genres,production_countries,spoken_languages
0,Toy Story,"[Animation, Comedy, Family]",[United States of America],[English]
1,Jumanji,"[Adventure, Fantasy, Family]",[United States of America],"[English, Français]"
2,Grumpier Old Men,"[Romance, Comedy]",[United States of America],[English]
3,Waiting to Exhale,"[Comedy, Drama, Romance]",[United States of America],[English]
4,Father of the Bride Part II,[Comedy],[United States of America],[English]


In [35]:
# Função para extrair a coleção (franquia) a partir do formato JSON/Dicionário

def extrair_colecao(texto):
    if pd.isna(texto):
        return np.nan
    try:
        dicionario = ast.literal_eval(texto)
        return dicionario['name'] # Retorna direto a string, não uma lista
    except:
        return np.nan

In [36]:
df_movies['belongs_to_collection'] = df_movies['belongs_to_collection'].apply(extrair_colecao)

In [37]:
# Visualizando o resultado limpo
display(df_movies[['title','belongs_to_collection', 'genres', 'production_countries', 'spoken_languages']].head())

,title,belongs_to_collection,genres,production_countries,spoken_languages
0,Toy Story,Toy Story Collection,"[Animation, Comedy, Family]",[United States of America],[English]
1,Jumanji,NaN,"[Adventure, Fantasy, Family]",[United States of America],"[English, Français]"
2,Grumpier Old Men,Grumpy Old Men Collection,"[Romance, Comedy]",[United States of America],[English]
3,Waiting to Exhale,NaN,"[Comedy, Drama, Romance]",[United States of America],[English]
4,Father of the Bride Part II,Father of the Bride Collection,[Comedy],[United States of America],[English]


---

## Criação de novas Colunas

In [38]:
# Criando uma coluna is_franchise para indicar se o filme pertence a uma franquia ou não
df_movies['is_franchise'] = df_movies['belongs_to_collection'].notna().astype(bool)
# Visualizando o resultado final
display(df_movies[['title','belongs_to_collection', 'is_franchise', 'genres', 'production_countries', 'spoken_languages']].head())

,title,belongs_to_collection,is_franchise,genres,production_countries,spoken_languages
0,Toy Story,Toy Story Collection,True,"[Animation, Comedy, Family]",[United States of America],[English]
1,Jumanji,NaN,False,"[Adventure, Fantasy, Family]",[United States of America],"[English, Français]"
2,Grumpier Old Men,Grumpy Old Men Collection,True,"[Romance, Comedy]",[United States of America],[English]
3,Waiting to Exhale,NaN,False,"[Comedy, Drama, Romance]",[United States of America],[English]
4,Father of the Bride Part II,Father of the Bride Collection,True,[Comedy],[United States of America],[English]


---
## Salvando Dataset antes de explodir as colunas

In [40]:
# Salvando o DataFrame processado para a próxima etapa de análise
caminho_saida = '../data/processed/movies_feature_engineered.csv'
df_movies.to_csv(caminho_saida, index=False)